# Adversarial Data Protection Framework - Experiment Notebook

Baseline vs protected training on CIFAR-10. Victim models are trained on protected train data and evaluated on a strictly clean test set.

In [ ]:
# Cell 1 - Setup (KHONG thay doi thu tu)
import subprocess
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"])

import os
import torch
import torchvision
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
os.makedirs("results/protected_samples", exist_ok=True)

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from src.datasets import get_cifar10
from src.evaluation import compute_attack_success_rate, compute_linf, compute_psnr, compute_ssim
from src.models import evaluate, get_victim_mobilenet, get_victim_resnet18, get_victim_vgg16, train_one_epoch
from src.techniques.unlearnable import apply_noise, generate_unlearnable_noise
from src.visualization import plot_before_after, plot_epsilon_vs_metrics, plot_technique_comparison

In [ ]:
SUBSET_SIZE = 1000  # increase to 5000 for the default Colab run
BATCH_SIZE = 128
BASELINE_EPOCHS = 3
VICTIM_EPOCHS = 3

train_loader, test_loader = get_cifar10(subset_size=SUBSET_SIZE, batch_size=BATCH_SIZE)
x_batch, y_batch = next(iter(train_loader))
print(x_batch.shape, x_batch.min().item(), x_batch.max().item())

In [ ]:
def train_classifier(model_fn, loader, epochs, lr=0.01):
    model = model_fn().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    criterion = nn.CrossEntropyLoss()
    history = []
    for epoch in range(epochs):
        loss, acc = train_one_epoch(model, loader, optimizer, criterion, device)
        history.append({"epoch": epoch + 1, "loss": loss, "train_acc": acc})
        print(history[-1])
    return model, history


def materialize_protected_loader(clean_loader, noise_dict, batch_size=BATCH_SIZE):
    xs, ys, clean_xs = [], [], []
    for x, y in clean_loader:
        x_protected = apply_noise(x.to(device), y.to(device), noise_dict).cpu()
        xs.append(x_protected)
        ys.append(y.cpu())
        clean_xs.append(x.cpu())
        del x_protected
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    clean_tensor = torch.cat(clean_xs, dim=0)
    protected_tensor = torch.cat(xs, dim=0).clamp(0, 1)
    label_tensor = torch.cat(ys, dim=0)
    loader = DataLoader(TensorDataset(protected_tensor, label_tensor), batch_size=batch_size, shuffle=True)
    return loader, clean_tensor, protected_tensor, label_tensor

In [ ]:
# Baseline: victim trained on clean train data and evaluated on clean test data.
baseline_model, baseline_history = train_classifier(
    lambda: get_victim_resnet18(device=device), train_loader, BASELINE_EPOCHS
)
baseline_clean_acc = evaluate(baseline_model, test_loader, device)
print({"baseline_clean_test_accuracy": baseline_clean_acc})

In [ ]:
# Unlearnable: generate class-wise noise, train victim on protected train data, evaluate on clean test data.
noise_dict = generate_unlearnable_noise(
    model_fn=lambda: get_victim_resnet18(device=device),
    train_loader=train_loader,
    epsilon=0.03,
    pgd_steps=5,
    inner_epochs=1,
    device=device,
)
torch.save(noise_dict, "results/unlearnable_noise_dict.pt")
print("Saved results/unlearnable_noise_dict.pt")
protected_loader, clean_train_tensor, protected_train_tensor, protected_labels = materialize_protected_loader(train_loader, noise_dict)

victim_model, victim_history = train_classifier(
    lambda: get_victim_resnet18(device=device), protected_loader, VICTIM_EPOCHS
)
clean_acc, asr = compute_attack_success_rate(victim_model, test_loader, device)
metrics = {
    "technique": "unlearnable",
    "victim_model": "ResNet-18",
    "epsilon": 0.03,
    "psnr": compute_psnr(clean_train_tensor, protected_train_tensor),
    "ssim": compute_ssim(clean_train_tensor, protected_train_tensor),
    "linf": compute_linf(clean_train_tensor, protected_train_tensor),
    "clean_test_accuracy": clean_acc,
    "asr": asr,
}
metrics

In [ ]:
# Save sample and before/after figure.
sample_idx = 0
fig = plot_before_after(clean_train_tensor[sample_idx], protected_train_tensor[sample_idx], "unlearnable")
fig.savefig("results/figures/before_after_unlearnable.png", dpi=150)
to_pil = torchvision.transforms.ToPILImage()
to_pil(clean_train_tensor[sample_idx]).save("results/protected_samples/original_unlearnable.png")
to_pil(protected_train_tensor[sample_idx]).save("results/protected_samples/protected_unlearnable.png")

In [ ]:
# Full ablation: 3 techniques x 5 epsilon x 3 victim models.
# Set FULL_ABLATION=True on Colab for the complete grid; False for a quick smoke run.
import torch.nn.functional as F
from src.models import get_surrogate_resnet50
from src.techniques.cloaking import cloak_images
from src.techniques.nightshade import load_clip_model, poison_images

FULL_ABLATION = False
EPSILON_VALUES = [0.01, 0.03, 0.05, 0.08, 0.1]
VICTIM_MODELS = {
    "ResNet-18": lambda: get_victim_resnet18(device=device),
    "VGG-16": lambda: get_victim_vgg16(device=device),
    "MobileNet": lambda: get_victim_mobilenet(device=device),
}
TECHNIQUES = ["unlearnable", "general_cloaking", "concept_poisoning"] if FULL_ABLATION else ["unlearnable"]
EPS_RUNS = EPSILON_VALUES if FULL_ABLATION else EPSILON_VALUES[:2]
VICTIM_RUNS = VICTIM_MODELS if FULL_ABLATION else {"ResNet-18": VICTIM_MODELS["ResNet-18"]}


def protect_cifar_upscale(technique_fn, x, batch_size=32):
    outputs = []
    for start in range(0, x.size(0), batch_size):
        batch = x[start : start + batch_size].to(device)
        up = F.interpolate(batch, size=(224, 224), mode="bilinear", align_corners=False)
        protected = technique_fn(up).detach()
        down = F.interpolate(protected, size=(32, 32), mode="bilinear", align_corners=False)
        outputs.append(down.cpu())
        del batch, up, protected, down
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return torch.cat(outputs, dim=0).clamp(0.0, 1.0)


def run_ablation(epsilon_values, victim_model_fns, techniques):
    clean_x, clean_y = [], []
    for x, y in train_loader:
        clean_x.append(x)
        clean_y.append(y)
    clean_x = torch.cat(clean_x, dim=0)
    clean_y = torch.cat(clean_y, dim=0)

    surrogate = get_surrogate_resnet50(device) if any(t in techniques for t in {"general_cloaking", "cloaking"}) else None
    clip_model = None
    if any(t in techniques for t in {"concept_poisoning", "nightshade"}):
        clip_model, _ = load_clip_model("ViT-B/32", device)

    records = []
    for technique in techniques:
        for eps in epsilon_values:
            print(f"[{technique}] eps={eps}")
            if technique == "unlearnable":
                noise = generate_unlearnable_noise(
                    model_fn=lambda: get_victim_resnet18(device=device),
                    train_loader=train_loader,
                    epsilon=eps,
                    pgd_steps=10 if FULL_ABLATION else 3,
                    inner_epochs=3 if FULL_ABLATION else 1,
                    device=device,
                )
                protected_x = apply_noise(clean_x.to(device), clean_y.to(device), noise).cpu()
            elif technique in {"general_cloaking", "cloaking"}:
                protected_x = protect_cifar_upscale(
                    lambda batch: cloak_images(
                        surrogate, batch, epsilon=eps,
                        pgd_steps=30 if FULL_ABLATION else 8, device=device,
                    ),
                    clean_x,
                )
            else:
                protected_x = protect_cifar_upscale(
                    lambda batch: poison_images(
                        clip_model, batch, target_concept="a photo of a cat",
                        epsilon=eps, pgd_steps=30 if FULL_ABLATION else 8, device=device,
                    ),
                    clean_x,
                )

            protected_loader = DataLoader(
                TensorDataset(protected_x, clean_y),
                batch_size=BATCH_SIZE,
                shuffle=True,
            )
            for model_name, model_fn in victim_model_fns.items():
                print(f"  victim={model_name}")
                victim, _ = train_classifier(
                    model_fn,
                    protected_loader,
                    epochs=3 if FULL_ABLATION else 1,
                )
                clean_acc, attack_sr = compute_attack_success_rate(victim, test_loader, device)
                records.append({
                    "technique": technique,
                    "victim_model": model_name,
                    "epsilon": round(eps, 4),
                    "psnr": compute_psnr(clean_x, protected_x),
                    "ssim": compute_ssim(clean_x, protected_x),
                    "linf": compute_linf(clean_x, protected_x),
                    "clean_test_accuracy": clean_acc,
                    "asr": attack_sr,
                })
                del victim
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    df = pd.DataFrame(records)
    df.to_csv("results/tables/ablation_results.csv", index=False)
    return df


ablation_df = run_ablation(EPS_RUNS, VICTIM_RUNS, TECHNIQUES)
ablation_df

In [ ]:
if len(ablation_df):
    plot_epsilon_vs_metrics(ablation_df, "unlearnable")
    if 0.05 in set(ablation_df["epsilon"]):
        plot_technique_comparison(ablation_df)

if torch.cuda.is_available():
    del baseline_model, victim_model, protected_train_tensor, clean_train_tensor
    torch.cuda.empty_cache()